In [1]:
import ccxt
import pandas as pd
import time
import os

# From Bitstamp exchange API

In [2]:
# — CONFIG — #
exchange = ccxt.bitstamp()
symbol = 'BTC/USD'
timeframe = '5m'

since = exchange.parse8601('2016-01-01T00:00:00Z')

# >>> END DATE LIMIT (31 Dec 2025) <<<
end_ts = exchange.parse8601('2025-12-31T23:59:59Z')

batch_limit = 1000
sleep_time = 0.5

output_dir = r"C:\Users\YILMAZ\Desktop\crypto_data"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "BTCUSD_5min_2016_2025.csv")

all_ohlcv = []
total_rows = 0
batch_count = 0

while since < end_ts:
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit=batch_limit)
    
    if not ohlcv:
        break

    # cutoff safeguard (in case API returns extra)
    ohlcv = [row for row in ohlcv if row[0] <= end_ts]
    if not ohlcv:
        break

    all_ohlcv += ohlcv
    batch_count += 1
    total_rows += len(ohlcv)

    last_timestamp = pd.to_datetime(ohlcv[-1][0], unit='ms')

    print(
        f"Batch {batch_count}: {len(ohlcv)} rows downloaded, "
        f"total rows = {total_rows}, last date = {last_timestamp}"
    )

    since = ohlcv[-1][0] + 1
    time.sleep(sleep_time)

# create DataFrame
df = pd.DataFrame(all_ohlcv, columns=['timestamp','open','high','low','close','volume'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

# save as csv
df.to_csv(output_file, index=False)

print(f"The data is saved as CSV: {output_file}")
print(f"Total rows: {len(df)}")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")

Batch 1: 1000 rows downloaded, total rows = 1000, last date = 2016-01-04 11:15:00
Batch 2: 999 rows downloaded, total rows = 1999, last date = 2016-01-07 22:30:00
Batch 3: 999 rows downloaded, total rows = 2998, last date = 2016-01-11 09:45:00
Batch 4: 999 rows downloaded, total rows = 3997, last date = 2016-01-14 21:00:00
Batch 5: 999 rows downloaded, total rows = 4996, last date = 2016-01-18 08:15:00
Batch 6: 999 rows downloaded, total rows = 5995, last date = 2016-01-21 19:30:00
Batch 7: 999 rows downloaded, total rows = 6994, last date = 2016-01-25 06:45:00
Batch 8: 999 rows downloaded, total rows = 7993, last date = 2016-01-28 18:00:00
Batch 9: 999 rows downloaded, total rows = 8992, last date = 2016-02-01 05:15:00
Batch 10: 999 rows downloaded, total rows = 9991, last date = 2016-02-04 16:30:00
Batch 11: 999 rows downloaded, total rows = 10990, last date = 2016-02-08 03:45:00
Batch 12: 999 rows downloaded, total rows = 11989, last date = 2016-02-11 15:00:00
Batch 13: 999 rows dow